# שבוע 9: ניתוח גרזני כנפיים מלא — מטלה 4

שבוע זה הוא שבוע **מטלה 4 — ניתוח EFA של גרזני כנפיים**.

הצינור המלא:
1. טעינת נתוני EFA
2. PCA על מקדמי EFA
3. ויזואליזציית מרחב הצורות
4. שחזור צורות ממוצע לכל קבוצה
5. בדיקה סטטיסטית (MANOVA בפרמוטציה)
6. בדיקת אלומטריה

> **טיפ**: שמרו את תמונות הגרפים שלכם (`plt.savefig`) לדוח.

In [ ]:
!pip install pyefd python-bidi -q
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
rtl = get_display
print('הכל מוכן!')

In [ ]:
import urllib.request

def load_efa_csv(url):
    with urllib.request.urlopen(url) as r:
        lines = r.read().decode('utf-8').strip().split('\n')
    data, groups = [], []
    for line in lines[1:]:
        parts = line.strip().split(',')
        groups.append(parts[0])
        data.append([float(x) for x in parts[1:]])
    return np.array(data), np.array(groups)

base = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/axes/'
try:
    efa_data, groups = load_efa_csv(base + 'axes_efa.csv')
    n_harmonics = efa_data.shape[1] // 4
    print(f'נטענו {len(efa_data)} גרזנים, {n_harmonics} הרמוניות')
    print(f'G3: {np.sum(groups=="G3")}, G4: {np.sum(groups=="G4")}')
except Exception as e:
    print(f'משתמשים בנתוני דוגמה: {e}')
    np.random.seed(42)
    n_g3, n_g4 = 30, 35
    base_g3 = np.random.randn(40) * 0.1
    base_g4 = base_g3 + np.random.randn(40) * 0.05 + 0.18
    efa_data = np.vstack([
        base_g3 + np.random.randn(n_g3, 40) * 0.08,
        base_g4 + np.random.randn(n_g4, 40) * 0.08
    ])
    groups = np.array(['G3'] * n_g3 + ['G4'] * n_g4)
    n_harmonics = 10

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
scores = pca.fit_transform(efa_data)
var = pca.explained_variance_ratio_ * 100

print('שונות מוסברת:')
for i in range(5):
    print(f'  PC{i+1}: {var[i]:.1f}%')
print(f'  PC1+PC2: {sum(var[:2]):.1f}%')

colors = {'G3': '#9C27B0', 'G4': '#4CAF50'}
fig, ax = plt.subplots(figsize=(9, 7))
for group, color in colors.items():
    mask = groups == group
    ax.scatter(scores[mask, 0], scores[mask, 1], c=color, s=90, alpha=0.8,
               label=f'{group} (n={mask.sum()})', edgecolors='white')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel(rtl(f'PC1 ({var[0]:.1f}% שונות)'))
ax.set_ylabel(rtl(f'PC2 ({var[1]:.1f}% שונות)'))
ax.set_title(rtl('מרחב EFA — גרזני כנפיים (G3 / G4)'), fontsize=13)
ax.legend(title=rtl('קבוצה עיצובית'))
plt.tight_layout()
plt.savefig('axes_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('הגרף נשמר: axes_pca.png')

## שחזור צורות ממוצע

In [ ]:
import pyefd

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
group_colors = {'G3': '#9C27B0', 'G4': '#4CAF50'}

for ax, (group, color) in zip(axes, group_colors.items()):
    mask = groups == group
    mean_coeffs = efa_data[mask].mean(axis=0)
    try:
        coeffs_matrix = mean_coeffs[:n_harmonics*4].reshape(n_harmonics, 4)
        contour = pyefd.reconstruct_contour(coeffs_matrix, locus=(0, 0),
                                             num_points=300, harmonic=n_harmonics)
        ax.plot(contour[:, 0], contour[:, 1], color=color, linewidth=2.5)
        ax.fill(contour[:, 0], contour[:, 1], alpha=0.15, color=color)
    except Exception:
        # fallback: plot PC space mean as text
        ax.text(0.5, 0.5, rtl(f'שחזור {group}\nלא זמין'), ha='center', va='center',
               transform=ax.transAxes, fontsize=14, color=color)
    ax.set_aspect('equal')
    ax.set_title(rtl(f'צורת ממוצע — {group} (n={mask.sum()})'), fontsize=12)
    ax.axis('off')

plt.suptitle(rtl('השוואת צורות ממוצע — גרזני כנפיים'), fontsize=14)
plt.tight_layout()
plt.savefig('axes_mean_shapes.png', dpi=150, bbox_inches='tight')
plt.show()

## בדיקה סטטיסטית

In [ ]:
def permutation_manova(X, groups, n_perm=999, seed=42):
    np.random.seed(seed)
    def f_stat(X, g):
        unique_g = np.unique(g)
        gm = X.mean(axis=0)
        between = sum(np.sum(g==u) * np.sum((X[g==u].mean(0) - gm)**2) for u in unique_g)
        within  = sum(np.sum((X[g==u] - X[g==u].mean(0))**2) for u in unique_g)
        return between / within if within > 0 else 0
    obs = f_stat(X, groups)
    perm = [f_stat(X, np.random.permutation(groups)) for _ in range(n_perm)]
    p = (np.sum(np.array(perm) >= obs) + 1) / (n_perm + 1)
    return obs, p, obs / (obs + 1)

f_val, p_val, r2 = permutation_manova(scores[:, :4], groups)
print(f'MANOVA בפרמוטציה (999 ערבולים):')
print(f'  F = {f_val:.3f}')
print(f'  p = {p_val:.3f}')
print(f'  R² = {r2:.3f} ({r2*100:.1f}% שונות מוסברת ע"י קבוצה)')
if p_val < 0.05:
    print('→ G3 ו-G4 שונות סטטיסטית מובהקת (p < 0.05)')
else:
    print('→ אין הבדל מובהק בין G3 ל-G4 (p ≥ 0.05)')

## בדיקת אלומטריה

האם קיים קשר בין מיקום במרחב PC1 לבין מיקום ב-PC2?

In [ ]:
from scipy import stats

slope, intercept, r, p_allom, se = stats.linregress(scores[:, 0], scores[:, 1])

fig, ax = plt.subplots(figsize=(7, 5))
for group, color in colors.items():
    mask = groups == group
    ax.scatter(scores[mask, 0], scores[mask, 1], c=color, s=70, alpha=0.8, label=group)
x_line = np.linspace(scores[:, 0].min(), scores[:, 0].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'k--', alpha=0.6, linewidth=1.5,
        label=f'r={r:.3f}, p={p_allom:.3f}')
ax.set_xlabel(rtl('PC1'))
ax.set_ylabel(rtl('PC2'))
ax.set_title(rtl('בדיקת קורלציה PC1-PC2 (אלומטריה)'))
ax.legend()
plt.tight_layout()
plt.show()

if p_allom < 0.05:
    print(f'נמצאה קורלציה מובהקת: r={r:.3f}, p={p_allom:.3f}')
else:
    print(f'לא נמצאה קורלציה מובהקת: r={r:.3f}, p={p_allom:.3f}')

## שאלות לדוח

1. כמה אחוז שונות מוסבר על ידי PC1 ו-PC2 יחד בניתוח EFA?
2. האם ההפרדה בין G3 ל-G4 מובהקת? (p-value, R²)
3. תארו בשפה פשוטה: מה ההבדל בצורת הגרזן בין G3 ל-G4?
4. מה המגבלות של שיטת EFA לעומת ניתוח ציוני דרך?